# api_request (GDAL / BAG v1 version)

Amsterdam API description: https://api.data.amsterdam.nl/api/

| Function | What it does |
|---|---|
| `get_pano_id` | panorama IDs for a bounding box and mission year (follows the paged results) |
| `get_pano_image` | image URL, camera location in RD, and camera location in WGS84 |
| `get_buildings_in_range` | BAG buildings (`panden`) within a radius of a point |
| `get_building_polygons` | footprint polygon per `pand_id` |
| `download_pano_image` | download a panorama as a PIL image |

**Requirements:** `requests`, `pillow` and `pyproj`, which are all in `requirements.txt`. This version uses `pyproj` for the coordinate conversion, so GDAL (`osgeo`) is not needed.

**Notes**

- `get_pano_image` returns **three** values `(image_url, (x_rd, y_rd), (lon, lat))`. Code that expects two values, such as the `run` notebook, needs `image_url, observer, _ = get_pano_image(pano_id)`.
- The transformer is created with `always_xy=True`, so points are always given as `(longitude, latitude)` and `(x, y)`. This avoids the latitude/longitude axis-order problem that GDAL 3 has with EPSG:4326.
- **BAG endpoint changed:** the buildings are now requested from `/v1/bag/panden`. The old code used `/v1/bag/v1/panden`, which no longer returns a `_embedded` key, so it failed with `KeyError: '_embedded'`. The `geometrie[within]=POINT(x y),d` filter is unchanged. The request now also sends `Accept-Crs`/`Content-Crs` headers (RD) and raises a readable error if the API answers with an error.
- The Amsterdam data platform has announced that API keys will become mandatory for its APIs (request one at https://keys.api.data.amsterdam.nl/clients/v1/). If you get a 401 or 403 error, that is the likely reason.
- The cells that call the live APIs need an internet connection, and the Amsterdam endpoints change from time to time.

In [ ]:
from io import BytesIO
import json
import requests
from PIL import Image
from pyproj import Transformer

API_TIMEOUT = 10

# WGS84 (EPSG:4326, lon/lat) to Rijksdriehoek (EPSG:28992, metres); always_xy keeps the order (lon, lat) / (x, y)
wgs84_to_rd = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)

In [ ]:
def get_pano_id(mission_year, bbox):
    """
    Get panoramic image id's for a user defined bounding box region
    """
    pano_url = f"https://api.data.amsterdam.nl/panorama/panoramas/?tags=mission-{mission_year}&bbox={bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}&srid=28992"

    with requests.get(pano_url, timeout=API_TIMEOUT) as response:
        pano_data_all = json.loads(response.content)

    pano_data = pano_data_all['_embedded']['panoramas']

    pano_id = []
    for item in pano_data:
        pano_id.append(item['pano_id'])

    # Check for next page with data
    next_page = pano_data_all['_links']['next']['href']

    # Exit the while loop if there is no next page
    while next_page:
        with requests.get(next_page, timeout=API_TIMEOUT) as response:
            pano_data_all = json.loads(response.content)

        pano_data = pano_data_all['_embedded']['panoramas']

        # Append the panorama id's to the list
        for item in pano_data:
            pano_id.append(item['pano_id'])

        # Check for next page
        next_page = pano_data_all['_links']['next']['href']

    return pano_id

In [ ]:
def get_pano_image(pano_id):
    """
    Get a panoramic image and its initial location coordinates
    and convert it from WGS84 (EPSG:4326) to Rijksdriehoek (EPSG:28992)
    """
    pano_url = f"https://api.data.amsterdam.nl/panorama/panoramas/{pano_id}/"
    with requests.get(pano_url, timeout=API_TIMEOUT) as response:
        pano_data = json.loads(response.content)

    geom = pano_data['geometry']['coordinates']

    # WGS84 to Rijksdriehoek (RD) conversion
    lon, lat = geom[0], geom[1]
    x_rd, y_rd = wgs84_to_rd.transform(lon, lat)

    # Get image_url from API
    image_url = pano_data['_links']['equirectangular_medium']['href']

    return (
        str(image_url),
        (x_rd, y_rd),
        (lon, lat),
    )

In [ ]:
# Buildings (BAG)
def get_buildings_in_range(observer, radius):
    """
    For a panoramic image, get the buildings that are whitin a user defined range.
    """
    bag_url = "https://api.data.amsterdam.nl/v1/bag/panden"
    # POINT(x y),d with x, y in RD metres and d the distance in metres
    params = {"geometrie[within]": f"POINT({observer[0]} {observer[1]}),{radius}"}
    headers = {"Accept-Crs": "EPSG:28992", "Content-Crs": "EPSG:28992"}

    with requests.get(bag_url, params=params, headers=headers, timeout=API_TIMEOUT) as response:
        if not response.ok:
            raise RuntimeError(f"BAG request failed ({response.status_code}): {response.text[:300]}")
        bag_data = json.loads(response.content)["_embedded"]["panden"]

    return bag_data

def get_building_polygons(pand_json):
    """ Get the polygon of a building """
    building_polygons = {}

    # Loop over json content
    for item in pand_json:
        pand_id = item["identificatie"]
        pand_polygon = item["geometrie"]["coordinates"][0]

        # A polygon should have at least 3 vertices to be a valid polygon
        if(len(pand_polygon) > 2 and pand_id):
            # Verify counter-clockwise winding order
            # TODO (Removed this because it happens rarely and slows the process down)

            building_polygons[pand_id] = pand_polygon

    return building_polygons

In [ ]:
## Download a panorama
def download_pano_image(pano_url):
    """ Download a panoramic image from the API """
    try:
        response = requests.get(pano_url, timeout=API_TIMEOUT)
        source_image = Image.open(BytesIO(response.content))
    except requests.exceptions.RequestException:
        print('HTTP Request failed. Aborting.')
        return

    return source_image

In [ ]:
# test API

BBOX_REGION = [122300.00, 486628.78, 122400.00, 486475.48]
MISSION_YEAR = 2018

pano_ids = get_pano_id(MISSION_YEAR, BBOX_REGION)
print(len(pano_ids), "panoramas, first:", pano_ids[:3])

In [ ]:
image_url, observer, observer_wgs84 = get_pano_image(pano_ids[0])
print("image url:     ", image_url)
print("RD location:   ", observer)
print("WGS84 location:", observer_wgs84)
# Sanity check: the RD point should be near the bounding box (x ~ 122300-122400, y ~ 486475-486629)

In [ ]:
pand_json = get_buildings_in_range(observer, 10)
building_polygons = get_building_polygons(pand_json)
print(len(building_polygons), "buildings in range:", list(building_polygons)[:5])

In [ ]:
import matplotlib.pyplot as plt

pano = download_pano_image(image_url)
if pano is not None:
    print("panorama size:", pano.size)
    plt.figure(figsize=(10, 3)); plt.imshow(pano); plt.axis("off"); plt.show()